In [39]:
using Pkg
Pkg.activate("../")
using PyCall
using Roots
using Plots
using NLsolve
using Optim
using LaTeXStrings

  Activating project at `~/Documents/Workshops/Surrogate/TwoVariableInter/3cpn_phasediag`


In [10]:
@kwdef  struct Params
    αA::Float64
    αB::Float64
    χN::Float64
end

Params

In [11]:
params = Params(αA=1.0, αB=0.5, χN=5.0)

Params(1.0, 0.5, 5.0)

In [33]:
# CE free energy (2 component)
Fc(ϕA, ϕB, αA, αB, χN) = ϕA*log(ϕA)/αA + ϕB*log(ϕB)/αB + χN*ϕA*ϕB
# F_c(ϕA, αB, χN) = F_c(ϕA, 1-ϕA, 1, αB, χN)
Fc(ϕA, params::Params) = Fc(ϕA, 1-ϕA, params.αA, params.αB, params.χN)


ϕ_to_μ(ϕA, ϕB, αA, αB, χN) = (1 + log(ϕA))/αA - (1 + log(1-ϕA))/αB + χN*(1-2ϕA)
# ϕ_to_μ(ϕ, α, χN) = ϕ_to_μ(ϕ, 1, α, χN)
ϕ_to_μ(ϕA, params::Params) = ϕ_to_μ(ϕA, 1-ϕA, params.αA, params.αB, params.χN)

function ϕ_to_Fg(ϕA, params::Params)
    return Fc(ϕA, params) - ϕ_to_μ(ϕA, params) * ϕA
end

# spinodal value, which equals to d2F_cdϕ2
spinodal(ϕA, ϕB, αA, αB, χN) = 1/(αA*ϕA) + 1/(αB*(1-ϕA)) - 2χN
# spinodal(ϕA, αB, χN) = spinodal(ϕA, 1, αB, χN)
spinodal(ϕA, params::Params) = spinodal(ϕA, 1-ϕA, params.αA, params.αB, params.χN)

# find spinodal points which let spinodal(ϕ) -> 0
# note that there are 2 spinodal points ϕ[1], ϕ[2]
function find_spinodal_points(params::Params)
    spinodal_to_zero = ϕs -> spinodal(ϕs, params)
    return Roots.find_zeros(spinodal_to_zero, 0, 1)
end

# F_g(μ_α): 在α相（左侧）实现 μ -> F_g的映射。在映射前，需要先通过 μ 得到α相的ϕ_α
# analytical format， whcih used in non surrogate method
function μtoFg_α(μ0, spinodal, params)
    diff = ϕ -> (ϕ_to_μ(ϕ, params) - μ0) # 目标残差：μ0 - μ(ϕ)
    ϕ_α = Roots.find_zero(diff, (0,spinodal[1]))  # 在左侧(0, spinodal[1])区间搜索根
    return ϕ_to_Fg(ϕ_α, params)
end

# F_g(μ_β): 在 β 相（右侧）实现 μ -> F_g的映射。在映射前，需要先通过 μ 得到β相的ϕ_β
# analytical format， whcih used in non surrogate method
function μtoFg_β(μ0, spinodal, params)
    diff = ϕ -> (ϕ_to_μ(ϕ, params) - μ0) # 目标残差：μ0 - μ(ϕ)
    ϕ_β = Roots.find_zero(diff, (spinodal[end], 1))  # 在右侧(spinodal[2], 1)区间搜索根
    return ϕ_to_Fg(ϕ_β, params)
end

# alfter finding μ satisfied F_g^α(μ) = F_g^β(μ), we map μ to 2 binodal points ϕs[1], ϕs[2]
function μ_to_ϕ(μ0, params::Params)
    diff = ϕ -> (ϕ_to_μ(ϕ, params) - μ0)
    ϕs = Roots.find_zeros(diff, 0, 1)
    return [ϕs[1], ϕs[end]]
end

# A wrapped Solver without surrogated method using root-finding algo
function non_surrogate_solve(params::Params)
    spinodal = find_spinodal_points(params)
    μα = ϕ_to_μ(spinodal[1], params)
	μβ = ϕ_to_μ(spinodal[end], params)

    Fg_diff = μ -> μtoFg_α(μ, spinodal, params) - μtoFg_β(μ, spinodal, params)
    μ_solved = Roots.find_zero(Fg_diff, (μα, μβ))
    ϕs_solved = μ_to_ϕ(μ_solved, params)

    return ϕs_solved
end


non_surrogate_solve (generic function with 1 method)

In [ ]:
ϕ_solved =  non_surrogate_solve(params)
Fg_α = ϕ_to_Fg(ϕ_solved[1], params)
Fg_β = ϕ_to_Fg(ϕ_solved[2], params)
println("ϕ_solved:  $ϕ_solved")
println("Fg in α disorder phase:  $Fg_α")
println("Fg in β disorder phase:  $Fg_β")
println("difference between 2 Fg:  $(abs(Fg_α - Fg_β))")

ϕ_solved:  [0.019250479290657332, 0.9273968777165841]
Fg in α disorder phase:  -0.017772981126123
Fg in β disorder phase:  -0.017772981126121454
difference between 2 Fg:  1.547373340571312e-15
